# TRI-FRAME: check that the datasets and saved results are in place

Run all cells. Put this notebook in the same folder as `TRI_FRAME_full.py` (your Downloads folder). It only reads files; it changes nothing.

In [ ]:
from pathlib import Path
import zipfile, json

HERE = Path.cwd()
DATA = HERE                              # datasets are searched for here, in subfolders and inside zip files
OUT = HERE.parent / "output" / "full"    # output/full next to the notebook folder
print("Notebook folder :", HERE)
print("Results folder  :", OUT, "(exists)" if OUT.exists() else "(NOT FOUND)")

In [ ]:
NEEDED = {
    "Adult":                [["adult.data"], ["adult.test"]],
    "ACS Income (CA 2018)": [["psam_p06.csv", "csv_pca.zip"]],
    "COMPAS":               [["compas-scores-two-years.csv"]],
    "South German Credit":  [["SouthGermanCredit.asc"]],
}

names = {}
for p in DATA.rglob("*"):
    if p.is_file():
        names.setdefault(p.name.lower(), str(p))
        if p.suffix.lower() == ".zip":
            try:
                for inner in zipfile.ZipFile(p).namelist():
                    names.setdefault(Path(inner).name.lower(), f"{p}  ->  {inner}")
            except zipfile.BadZipFile:
                pass

all_ok = True
for dataset, groups in NEEDED.items():
    for options in groups:
        hit = next((names[o.lower()] for o in options if o.lower() in names), None)
        all_ok &= bool(hit)
        print(f"{'OK     ' if hit else 'MISSING'} {dataset:22s} {' or '.join(options):32s} {hit or ''}")
print("\nAll datasets found." if all_ok else "\nSome datasets are missing: put them in this folder (zipped or unzipped).")

In [ ]:
print("Code:")
for f in ["TRI_FRAME_full.py", "TRI_FRAME_bounds.ipynb"]:
    print(f"  {'OK     ' if (HERE / f).exists() else 'MISSING'} {f}")

print("\nSaved results from the full run:")
models = list((OUT / "models").glob("*.pt")) if (OUT / "models").exists() else []
print(f"  {'OK     ' if models else 'MISSING'} models folder: {len(models)} saved models")
for d in ["adult", "acs_income", "compas", "south_german_credit"]:
    f = OUT / "tuning" / f"{d}_selected.json"
    print(f"  {'OK     ' if f.exists() else 'MISSING'} tuning/{f.name}")
f = OUT / "results_main.csv"
print(f"  {'OK     ' if f.exists() else 'MISSING'} results_main.csv")